In [0]:
from LDCDataAccessLayerPy import KeyVaultManager, SharePointManager, SqlManager, databricks_init
from datetime import datetime, timedelta
from LDCDataAccessLayerPy import databricks_init, DataLakeManagerGen2
from io import BytesIO
import LDCDataAccessLayerPy
#Initiate the secret to access KeyVault secrets
databricks_init(dbutils, 'GO')
sp_mgr = SharePointManager()
sql_mgr = SqlManager()

import logging
logger = spark._jvm.org.apache.log4j
logging.getLogger("py4j").setLevel(logging.ERROR)

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans
from openpyxl import load_workbook

from datetime import datetime, timedelta
import re

url = "https://ldcom365.sharepoint.com"



In [0]:
lineups=sp_mgr.read_pd_from_excel('/sites/GRP-TradingLineups/Lineups/PROVIDERS/OILS/lineup.xlsx',sheet_name='DailyVesselLineUp',header=None)

# Locate the row where 'Port' is located
port_row_index = lineups[lineups.eq('Port').any(axis=1)].index[0]

lineups.columns = lineups.iloc[port_row_index]

lineups = lineups.iloc[port_row_index+1:].reset_index(drop=True)


ferti = ["SULPHURIC ACID", 
        "MAP",
        "PHOSPHORIC ROCK",
        "FERTILIZER SOLID",
        "GRANULAR UREA",
        "TSP",
        "SODIUM METHYLATE",
        "UAN",
        "FERTILIZER SOLID",
        "OTHER FERTILIZER",
        "DAP",
        "K POTASSIUM",
        "MESZ",
        "GLYPHOSATE",
        "DAP",
        "MES9",
        'SODA ASH',
        "GMOP",
        "CAUSTIC SODA",
        'METHANOL'
        ]



to_exclude=[
    'GASOIL',"COAL","STEEL BAR","IRON ORE","WOOD LOGS","LUBOILS","LIMESTONE","ULSD","STEEL PIPES","CITRUS","PETCOKE","CHEMICALS SOLID","STEEL COILS","WOOD PULP","BUTANE GAS","ASPHALT",'ULSD (ULTRA LOW SULPHUR DIESEL)','BARITE'
]


ferti_lineups=lineups[lineups['Commodity'].isin(ferti)]

grains_lineups = lineups[
    ~lineups['Commodity'].isin(ferti) &
    ~lineups['Commodity'].isin(to_exclude) &
    lineups['Commodity'].notna() & 
    (lineups['Commodity'].astype(str).str.strip() != '')
].reset_index(drop=True)


ferti_lineups.reset_index(drop=True)
grains_lineups.reset_index(drop=True)

In [0]:
grains_lineups.Commodity.unique()

In [0]:
ferti_lineups_html = ferti_lineups.to_html(index=False)
grains_lineups_html = grains_lineups.to_html(index=False)

In [0]:
import pandas as pd
from io import BytesIO
from datetime import datetime

# Convert ferti_lineups DataFrame to Excel in memory
excel_buffer_ferti = BytesIO()
ferti_lineups.to_excel(excel_buffer_ferti, index=False, sheet_name='Lineups')
excel_buffer_ferti.seek(0)  # Reset the buffer's position

excel_buffer_grains = BytesIO()
grains_lineups.to_excel(excel_buffer_grains, index=False, sheet_name='Lineups')
excel_buffer_grains.seek(0)  # Reset the buffer's position


attachment = {"ferti_lineups.xlsx": excel_buffer_ferti}

html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Fertilizers and Agrochemicals Lineups NABSA </title>
    <style>
        .container {{
            justify-content: space-between;
            align-items: flex-start;
            width: 100%;
            gap: 10px;
        }}
        .chart-container, .table-container, .extra-container {{
            flex: 0;
            padding: 10px;
            height: 100%;
        }}
        .table-container {{
            text-align: center;
            margin: auto;
            flex: 1; 
            min-width: 300px; 
        }}
        table {{
            width: 100%;
            border-collapse: collapse;
        }}
        th, td {{
            border: 1px solid black;
            padding: 8px;
            text-align: center;
        }}
        th {{
            background-color: #f2f2f2;
        }}
        img {{
            width: 100%;
            height: auto;
            min-width: 750px; /* Set a minimum width */
            min-height: 750px; 
            object-fit: contain;
        }}
        .wide-table {{
            width: 100%;
            table-layout: fixed;
        }}

        .wide-table th, .wide-table td {{
            padding: 8px;
            width: 40%;
            min-width: 100px;
            white-space: nowrap;
        }}
        
    </style>
</head>
<body>
    <h1>Fertlizer and Agrochemicals Lineups</h1>
    <table class="layout">
        <tr>
            <!-- Table Section -->
            <td class="table-container">
                <h2>NABSA Fertilizers Lineups</h2>
                {ferti_lineups_html}  <!-- Insert DataFrame as an HTML table -->
            </td>
        </tr>
    </table>
</body>
</html>
"""
adress=['florian.girardi-ext@ldc.com,gustavo.ferramondo@ldc.com']
adress_test=['florian.girardi-ext@ldc.com']

# Send the email
LDCDataAccessLayerPy.mail.mail_send(
    to=adress_test,
    subject=f'Fertlizers and Agrochemicals Lineups {datetime.now().strftime("%d-%m")}',
    from_addr="florian.girardi-ext@ldc.com",
    body=html_content,
    mime_type="html",
    attachment=attachment
)


In [0]:


attachment = {"grains_lineups.xlsx": excel_buffer_grains}

html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Fertilizers and Agrochemicals Lineups NABSA </title>
    <style>
        .container {{
            justify-content: space-between;
            align-items: flex-start;
            width: 100%;
            gap: 10px;
        }}
        .chart-container, .table-container, .extra-container {{
            flex: 0;
            padding: 10px;
            height: 100%;
        }}
        .table-container {{
            text-align: center;
            margin: auto;
            flex: 1; 
            min-width: 300px; 
        }}
        table {{
            width: 100%;
            border-collapse: collapse;
        }}
        th, td {{
            border: 1px solid black;
            padding: 8px;
            text-align: center;
        }}
        th {{
            background-color: #f2f2f2;
        }}
        img {{
            width: 100%;
            height: auto;
            min-width: 750px; /* Set a minimum width */
            min-height: 750px; 
            object-fit: contain;
        }}
        .wide-table {{
            width: 100%;
            table-layout: fixed;
        }}

        .wide-table th, .wide-table td {{
            padding: 8px;
            width: 40%;
            min-width: 100px;
            white-space: nowrap;
        }}
        
    </style>
</head>
<body>
    <h1>Grains Lineups</h1>
    <table class="layout">
        <tr>
            <!-- Table Section -->
            <td class="table-container">
                <h2>NABSA Grains Lineups</h2>
                {grains_lineups_html}  <!-- Insert DataFrame as an HTML table -->
            </td>
        </tr>
    </table>
</body>
</html>
"""
adress=['florian.girardi-ext@ldc.com,gustavo.ferramondo@ldc.com']
adress_test=['florian.girardi-ext@ldc.com']

# Send the email
LDCDataAccessLayerPy.mail.mail_send(
    to=adress_test,
    subject=f'Grains Lineups {datetime.now().strftime("%d-%m")}',
    from_addr="florian.girardi-ext@ldc.com",
    body=html_content,
    mime_type="html",
    attachment=attachment
)
